# Pipeline de Limpieza — GFP Implementation Gap

**Descripción:** Limpieza y construcción de la base de datos para el modelo de brecha de implementación.
Cubre las tres modalidades: Contrata, Administración Directa (AD) y ARCC.

**Flujo:**
1. Configuración de modalidad
2. Carga y reshape de data
3. Filtros macro (modalidad, gobierno local, temporal)
4. Conversión de tipos
5. Filtros de calidad
6. Construcción de variable dependiente (brecha)
7. Creación de nuevas variables
8. Imputación y filtros de variabilidad
9. Codificación (dummies, log)
10. [Opcional] Filtro de correlación
11. Exportación

## 0. Configuración — cambiar aquí para cada modalidad

In [1]:
# ============================================================
# CONFIGURACIÓN PRINCIPAL
# Opciones de MODALIDAD: 'contrata' | 'ad' | 'arcc'
# ============================================================

MODALIDAD = 'arcc'

CONFIG = {
    'contrata': {
        'filtro_col': 'modalidad_ejecucion',
        'filtro_val': 'Contrata',
        'incluir_modalidad_como_feature': False,  # constante -> no aporta al modelo
        'sub_filtro_modalidad': None,
    },
    'ad': {
        'filtro_col': 'modalidad_ejecucion',
        'filtro_val': 'Administración directa',
        'incluir_modalidad_como_feature': False,
        'sub_filtro_modalidad': None,
    },
    'arcc': {
        'filtro_col': 'marca_reconstruccion',
        'filtro_val': 'Si',
        'incluir_modalidad_como_feature': True,   # el modelo ve qué modalidad tiene cada obra
        # None = todas las modalidades dentro de ARCC
        # 'Contrata' | 'Administración directa' para sub-análisis
        'sub_filtro_modalidad': None,
    },
}

cfg = CONFIG[MODALIDAD]

# Rutas — ajustar si cambia la ubicación del proyecto
PATH_DATA        = 'C:/15_GFP/data/raw/data.xlsx'
PATH_VARNAMES    = 'C:/15_GFP/data/dictionaries/var_names_of_proyectos_ejecucion.xlsx'
PATH_MACROZONA   = 'C:/15_GFP/data/dictionaries/macro_zona.xlsx'
PATH_OUTPUT      = f'C:/15_GFP/data/processed/{MODALIDAD}/1_data_{MODALIDAD}.xlsx'

# Sub-filtro de modalidad para ARCC (se aplica solo si está definido)
if MODALIDAD == 'arcc' and cfg['sub_filtro_modalidad']:
    sub_label = cfg['sub_filtro_modalidad'].lower().replace(' ', '_').replace('ó', 'o')
    PATH_OUTPUT = f'C:/15_GFP/data/processed/arcc/1_data_arcc_{sub_label}.xlsx'

print(f'Modalidad: {MODALIDAD.upper()}')
print(f'Filtro principal: {cfg["filtro_col"]} == "{cfg["filtro_val"]}"')
if cfg.get('sub_filtro_modalidad'):
    print(f'Sub-filtro modalidad: {cfg["sub_filtro_modalidad"]}')
print(f'Output: {PATH_OUTPUT}')

Modalidad: ARCC
Filtro principal: marca_reconstruccion == "Si"
Output: C:/15_GFP/data/processed/arcc/1_data_arcc.xlsx


## 1. Importaciones

In [2]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\luisv\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [3]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import re
import unicodedata
import os
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Carga y reshape de data

In [4]:
dataoriginal = pd.read_excel(PATH_DATA, engine='openpyxl')
data = dataoriginal.copy()

# Reshape: las primeras 2 filas son metadatos, la fila 3 contiene los nombres reales
data = data.iloc[2:].reset_index(drop=True)
data.columns = data.iloc[0]
data = data.iloc[1:].reset_index(drop=True)

# Columnas con nombre duplicado en posiciones fijas
data.columns.values[45] = 'M1'
data.columns.values[48] = 'M2'

print(f'Registros originales: {len(data):,}')

Registros originales: 170,340


In [5]:
# Cargar diccionario de variables
names = pd.read_excel(PATH_VARNAMES, engine='openpyxl')

# Eliminar columnas marcadas como NO
columnas_a_eliminar = names.loc[names['Resultado'] == 'NO', 'names'].tolist()
data = data.drop(columns=[col for col in columnas_a_eliminar if col in data.columns])

# Renombrar columnas según el diccionario
mapeo_nombres = dict(zip(names['names'], names['varname']))
data = data.rename(columns={col: mapeo_nombres[col] for col in data.columns if col in mapeo_nombres})

## 3. Filtros macro

### 3.1 Filtro por modalidad / marca

In [6]:
total_original = len(data)

# Filtro principal según modalidad configurada
data = data[data[cfg['filtro_col']] == cfg['filtro_val']]

# Sub-filtro de modalidad (solo aplica en ARCC si está definido)
if MODALIDAD == 'arcc' and cfg['sub_filtro_modalidad']:
    data = data[data['modalidad_ejecucion'] == cfg['sub_filtro_modalidad']]

# Filtro de nivel de gobierno
data = data[data['nivel_gobierno'] == 'GOBIERNO LOCAL']
data = data[data['sector_entidad'] == 'GOBIERNOS LOCALES']

print(f'Registros tras filtro {MODALIDAD.upper()}: {len(data):,} (excluidos: {total_original - len(data):,})')

Registros tras filtro ARCC: 3,601 (excluidos: 166,739)


### 3.2 Filtro temporal: obras con inicio entre 2018 y 2024

In [7]:
# Guardar fechas originales antes de parsear
data['fecha_inicio_obra_original']      = data['fecha_inicio_obra']
data['fecha_fin_programada_original']   = data['fecha_fin_programada']
data['fecha_finalizacion_real_original']= data['fecha_finalizacion_real']

# Parsear fechas
for col in ['fecha_inicio_obra', 'fecha_fin_programada', 'fecha_finalizacion_real']:
    data[col] = pd.to_datetime(data[col], errors='coerce', dayfirst=True)

# Conservar obras con inicio en 2018-2024 (o sin fecha de inicio)
n_antes = len(data)
data['filtro_fecha'] = data['fecha_inicio_obra'].dt.year.between(2018, 2024)
data = data[data['filtro_fecha'] | data['fecha_inicio_obra'].isna()].copy()

print(f'Registros tras filtro temporal: {len(data):,} (excluidos: {n_antes - len(data):,})')

Registros tras filtro temporal: 3,523 (excluidos: 78)


## 4. Conversión de tipos

In [8]:
# Limpiar nombres de columnas y metadata
data.columns  = data.columns.astype(str).str.strip()
names['varname'] = names['varname'].astype(str).str.strip()
names['Tipo']    = names['Tipo'].str.upper().str.strip()

# Variables con separadores decimales incorrectos (espacio en lugar de punto)
variables_problematicas = [
    'monto_viable', 'monto_expediente', 'porcentaje_terreno_entregado',
    'avance_fisico_programado', 'avance_fisico_real',
    'monto_valorizacion_programada', 'monto_valorizacion_ejecutada',
    'porcentaje_ejecucion_financiera', 'monto_ejecucion_financiera',
    'monto_cipr', 'monto_adicionales_obra', 'monto_adicionales_supervision',
    'monto_deductivos_obra', 'costo_obra_soles', 'monto_total_devengado',
    'monto_contrato_soles_1', 'monto_aprobado_soles', 'monto_contrato_soles_2'
]

def reemplazar_espacio_decimal(val):
    if isinstance(val, str):
        return re.sub(r'(?<=\d) (?=\d{1,4}$)', '.', val.strip())
    return val

# Convertir variables numéricas
num_vars = names[names['Tipo'] == 'NUM']['varname']
nan_report = []

for var in num_vars:
    if var in data.columns:
        try:
            serie = data[var]
            if isinstance(serie, pd.Series) and serie.ndim == 1:
                antes_nan    = serie.isna().sum()
                original_vals = serie.copy()
                if var in variables_problematicas:
                    serie = serie.apply(reemplazar_espacio_decimal)
                data[var]    = pd.to_numeric(serie, errors='coerce')
                despues_nan  = data[var].isna().sum()
                nuevos_nan   = despues_nan - antes_nan
                if nuevos_nan > 0:
                    nan_report.append({
                        'variable': var,
                        'na_antes': antes_nan,
                        'na_despues': despues_nan,
                        'na_nuevos': nuevos_nan,
                    })
        except:
            continue

# Convertir variables categóricas
tipos_categoricos = names[names['Tipo'].isin(['DICO', 'POLI'])]['varname']
for var in tipos_categoricos:
    if var in data.columns:
        try:
            data[var] = data[var].astype('category')
        except:
            pass

print('Conversión de tipos completada.')
if nan_report:
    print(f'Variables con nuevos NaN tras conversión numérica: {len(nan_report)}')

Conversión de tipos completada.


## 5. Filtros de calidad

In [9]:
n_ini = len(data)

# 5.1 Obras con 0% de avance y más de 5 años de antigüedad
limite_5anhos = pd.Timestamp.now() - pd.DateOffset(years=5)
data = data[~((data['avance_fisico_real'] == 0) & (data['fecha_inicio_obra'] < limite_5anhos))]
print(f'Tras filtro 0-avance >5 años: {len(data):,} (excluidos: {n_ini - len(data):,})')

# 5.2 Incoherencia: >95% avance y estado != Finalizado
n = len(data)
data = data[~((data['avance_fisico_real'] >= 95) & (data['estado_ejecucion'] != 'Finalizado'))]
print(f'Tras filtro avance-estado incoherente: {len(data):,} (excluidos: {n - len(data):,})')

# 5.3 CUI y SNIP válidos
n = len(data)
data['codigo_inversion'] = data['codigo_inversion'].astype(str).str.strip()
data['codigo_snip']      = data['codigo_snip'].astype(str).str.strip()
data = data[
    data['codigo_inversion'].notna() & (data['codigo_inversion'] != '0') &
    data['codigo_snip'].notna()      & (data['codigo_snip'] != '0')
]
print(f'Tras filtro CUI/SNIP válidos: {len(data):,} (excluidos: {n - len(data):,})')

# 5.4 Excluir obras que no son infraestructura (por nombre)
n = len(data)
palabras_clave_excluir = [
    'camion', 'camiones', 'vehiculo', 'vehículos', 'adquisicion', 'adquisición',
    'equipamiento', 'mobiliario', 'charla', 'charlas', 'sensibilizacion', 'capacitación',
    'computadoras', 'kits', 'materiales', 'herramientas', 'instrumentos', 'impresora'
]
data['nombre_obra_limpio'] = data['nombre_obra'].astype(str).fillna('').str.lower()
regex_excluir = '|'.join([re.escape(p) for p in palabras_clave_excluir])
data = data[~data['nombre_obra_limpio'].str.contains(regex_excluir, na=False, regex=True)].copy()
print(f'Tras filtro nombre de obra: {len(data):,} (excluidos: {n - len(data):,})')

Tras filtro 0-avance >5 años: 3,519 (excluidos: 4)
Tras filtro avance-estado incoherente: 3,450 (excluidos: 69)
Tras filtro CUI/SNIP válidos: 3,419 (excluidos: 31)
Tras filtro nombre de obra: 3,400 (excluidos: 19)


## 6. Variable dependiente: Implementation Gap (brecha)

### 6.1 Filtros sobre fechas para la variable dependiente

In [10]:
fecha_descarga = pd.to_datetime('2024-10-28')
n_ini = len(data)

# Sin fecha de finalización real → no se puede calcular brecha
data = data[data['fecha_finalizacion_real'].notna()].copy()
print(f'Con fecha fin real: {len(data):,} (excluidos: {n_ini - len(data):,})')

# Fecha programada anterior a fecha de inicio
n = len(data)
mask_prog_inv = (
    (data['fecha_fin_programada'] < data['fecha_inicio_obra']) &
    data['fecha_fin_programada'].notna() &
    data['fecha_inicio_obra'].notna()
)
data = data[~mask_prog_inv].copy()
print(f'Tras filtro fecha-prog < fecha-inicio: {len(data):,} (excluidos: {n - len(data):,})')

# Fecha real anterior a fecha de inicio
n = len(data)
mask_real_inv = (
    (data['fecha_finalizacion_real'] < data['fecha_inicio_obra']) &
    data['fecha_finalizacion_real'].notna() &
    data['fecha_inicio_obra'].notna()
)
data = data[~mask_real_inv].copy()
print(f'Tras filtro fecha-real < fecha-inicio: {len(data):,} (excluidos: {n - len(data):,})')

# Fecha real posterior a la fecha de descarga
n = len(data)
data = data[data['fecha_finalizacion_real'] <= fecha_descarga].copy()
print(f'Tras filtro fecha-real > corte: {len(data):,} (excluidos: {n - len(data):,})')

# Fechas programadas inválidas (nulas o >2025)
n = len(data)
data = data[
    data['fecha_fin_programada'].notna() &
    (data['fecha_fin_programada'].dt.year <= 2025)
].copy()
print(f'Tras filtro fecha-prog inválida: {len(data):,} (excluidos: {n - len(data):,})')

Con fecha fin real: 2,644 (excluidos: 756)
Tras filtro fecha-prog < fecha-inicio: 2,644 (excluidos: 0)
Tras filtro fecha-real < fecha-inicio: 2,634 (excluidos: 10)
Tras filtro fecha-real > corte: 2,631 (excluidos: 3)
Tras filtro fecha-prog inválida: 2,631 (excluidos: 0)


### 6.2 Cálculo de brecha

In [11]:
mask_validas = data['fecha_fin_programada'].notna()
data['brecha_dias'] = pd.NA

data.loc[mask_validas, 'brecha_dias'] = (
    data.loc[mask_validas, 'fecha_finalizacion_real'] -
    data.loc[mask_validas, 'fecha_fin_programada']
).dt.days

data['brecha_existente'] = pd.NA
data.loc[mask_validas & (data['brecha_dias'] > 0), 'brecha_existente'] = 'Sí'
data.loc[mask_validas & (data['brecha_dias'] <= 0), 'brecha_existente'] = 'No'

print(data['brecha_existente'].value_counts())
print(f'\nBrecha media (días): {data["brecha_dias"].mean():.0f}')

brecha_existente
Sí    1523
No    1108
Name: count, dtype: int64

Brecha media (días): 67


## 7. Nuevas variables

In [12]:
# 7.1 Macrozona geográfica
macrozona = pd.read_excel(PATH_MACROZONA, engine='openpyxl')

def asignar_region(row):
    if row['COSTA'] == 1 and row['COS_NOR'] == 1:
        return 'costa norte'
    elif row['COSTA'] == 1:
        return 'costa centro/sur'
    elif row['SIERRA'] == 1 and row['SIER_SUR'] == 1:
        return 'sierra sur'
    elif row['SIERRA'] == 1:
        return 'sierra centro/norte'
    elif row['SELVA'] == 1:
        return 'selva'
    else:
        return 'no clasificado'

macrozona['Region'] = macrozona.apply(asignar_region, axis=1)
macrozona = macrozona[['NOMPRO', 'Region']]

# 7.2 Normalización de nombres de provincia para merge
def normalizar_nombre(nombre):
    if pd.isna(nombre):
        return ''
    nombre = nombre.upper()
    nombre = unicodedata.normalize('NFKD', nombre).encode('ASCII', 'ignore').decode('utf-8')
    nombre = re.sub(r'[^A-Z\s]', '', nombre)
    return nombre.strip()

macrozona['provincia_norm'] = macrozona['NOMPRO'].apply(normalizar_nombre)
data['provincia_norm']      = data['provincia'].apply(normalizar_nombre)

n_antes = len(data)
data = data.merge(macrozona, on='provincia_norm', how='inner')
print(f'Tras merge macrozona: {len(data):,} (excluidos sin match: {n_antes - len(data):,})')
print(data['Region'].value_counts())

Tras merge macrozona: 2,539 (excluidos sin match: 92)


Region
sierra centro/norte    1084
costa norte             881
sierra sur              377
costa centro/sur        190
selva                     7
Name: count, dtype: int64


In [13]:
# 7.3 Tipo de obra concatenado
data['tipo_obra_full'] = (
    data['tipo_obra_nivel1'].astype(str) + '_' +
    data['tipo_obra_nivel2'].astype(str) + '_' +
    data['tipo_obra_nivel3'].astype(str)
)

# 7.4 Año de inicio de obra
data['anio_inicio_obra'] = data['fecha_inicio_obra'].dt.year

# 7.5 Log del monto aprobado
data['log_monto_aprobado'] = np.log1p(data['monto_aprobado_soles'])

## 8. Estadísticas descriptivas

In [14]:
# Convertir categorías de brecha y region
data[['brecha_existente', 'Region']] = data[['brecha_existente', 'Region']].astype('category')

# Descriptivos numéricos
data['brecha_dias'] = pd.to_numeric(data['brecha_dias'], errors='coerce')
variables_num = data.select_dtypes(include=['number']).columns.tolist()

tabla_descriptiva = data[variables_num].describe().T
tabla_descriptiva['moda']     = data[variables_num].mode().iloc[0]
tabla_descriptiva['varianza'] = data[variables_num].var()
tabla_descriptiva['q1']       = data[variables_num].quantile(0.25)
tabla_descriptiva['q3']       = data[variables_num].quantile(0.75)
tabla_descriptiva['rango']    = tabla_descriptiva['max'] - tabla_descriptiva['min']
tabla_descriptiva['coef_var'] = tabla_descriptiva['std'] / tabla_descriptiva['mean']

tabla_descriptiva

,count,mean,std,min,25%,50%,75%,max,moda,varianza,q1,q3,rango,coef_var
n_informes_monitores,2539.0,0.000000e+00,0.000000e+00,0.00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,NaN
n_denuncias,2539.0,7.089405e-02,3.709502e-01,0.00,0.000000,0.000000,0.000000e+00,7.000000e+00,0.0,1.376040e-01,0.000000,0.000000e+00,7.000000e+00,5.232459
n_informes_control,2539.0,8.263096e-01,1.589763e+00,0.00,0.000000,0.000000,1.000000e+00,1.500000e+01,0.0,2.527345e+00,0.000000,1.000000e+00,1.500000e+01,1.923931
n_comentarios_ciudadanos,2539.0,7.877117e-04,2.806067e-02,0.00,0.000000,0.000000,0.000000e+00,1.000000e+00,0.0,7.874013e-04,0.000000,0.000000e+00,1.000000e+00,35.623023
n_obras_relacionadas,2539.0,1.088618e+00,3.444145e-01,1.00,1.000000,1.000000,1.000000e+00,5.000000e+00,1.0,1.186214e-01,1.000000,1.000000e+00,4.000000e+00,0.316378
monto_aprobado_soles,2528.0,3.093124e+06,6.099559e+07,0.00,205350.665000,732579.070000,2.177266e+06,3.056543e+09,0.0,3.720462e+15,205350.665000,2.177266e+06,3.056543e+09,19.719739
plazo_ejecucion_dias,2539.0,9.108980e+01,5.961227e+01,2.00,52.500000,75.000000,1.200000e+02,4.500000e+02,60.0,3.553623e+03,52.500000,1.200000e+02,4.480000e+02,0.654434
porcentaje_terreno_entregado,2509.0,9.998007e+01,9.982048e-01,50.00,100.000000,100.000000,1.000000e+02,1.000000e+02,100.0,9.964129e-01,100.000000,1.000000e+02,5.000000e+01,0.009984
avance_fisico_real,2539.0,9.590610e+01,1.160630e+01,0.58,97.870000,100.000000,1.000000e+02,1.000000e+02,100.0,1.347062e+02,97.870000,1.000000e+02,9.942000e+01,0.121017
porcentaje_ejecucion_financiera,2539.0,9.072801e+01,1.955317e+02,0.00,81.020000,90.820000,9.788000e+01,7.104240e+03,100.0,3.823266e+04,81.020000,9.788000e+01,7.104240e+03,2.155142


In [15]:
# Reporte de nulos
nulos = data.isna().sum()
porcentaje_nulos = (nulos / len(data)) * 100
df_nulos = pd.DataFrame({
    'Variable': data.columns,
    'Nulos': nulos.values,
    'Porcentaje_Nulos': porcentaje_nulos.values
}).query('Nulos > 0').sort_values('Porcentaje_Nulos', ascending=False)

df_nulos

,Variable,Nulos,Porcentaje_Nulos
11,otra_marca,2539,100.000000
39,tipo_certificado_inversion,2539,100.000000
47,transferencia_otro_entidad,2522,99.330445
48,tipo_entidad_recepciona,2522,99.330445
34,causal_paralizacion,2519,99.212288
35,n_dias_paralizado,2519,99.212288
36,comentarios,2263,89.129579
22,tipo_formato,1316,51.831430
21,estado_proyecto,1259,49.586451
31,porcentaje_terreno_entregado,30,1.181568


## 9. Preparación para el modelo

In [16]:
# 9.1 Eliminar columnas de fecha y auxiliares
columnas_fecha = names.loc[names['Resultado'] == 'FECHA', 'names'].tolist()
data = data.drop(columns=[col for col in columnas_fecha if col in data.columns], errors='ignore')

columnas_aux = [
    'fecha_inicio_obra', 'fecha_fin_programada', 'fecha_finalizacion_real',
    'fecha_inicio_obra_original', 'fecha_fin_programada_original',
    'fecha_finalizacion_real_original', 'filtro_fecha', 'nombre_obra_limpio'
]
data = data.drop(columns=columnas_aux, errors='ignore')

# 9.2 Excluir solo casos sin variable dependiente
data = data[data['brecha_existente'].notna()].copy().reset_index(drop=True)
print(f'Registros con brecha definida: {len(data):,}')

Registros con brecha definida: 2,539


In [17]:
# 9.3 Filtro de missings: eliminar columnas con >10% de nulos
threshold = 0.10
missing_ratio = data.isna().mean()
cols_excluidas_missing = missing_ratio[missing_ratio > threshold].index.tolist()
data = data.loc[:, missing_ratio <= threshold]
print(f'Columnas eliminadas por >10% missings: {len(cols_excluidas_missing)}')
print(cols_excluidas_missing)

# 9.4 Requiere distrito no nulo para imputación geográfica
data = data[data['distrito'].notna()].copy()

Columnas eliminadas por >10% missings: 9
['otra_marca', 'estado_proyecto', 'tipo_formato', 'causal_paralizacion', 'n_dias_paralizado', 'comentarios', 'tipo_certificado_inversion', 'transferencia_otro_entidad', 'tipo_entidad_recepciona']


In [18]:
# 9.5 Imputación geográfica (departamento → provincia → distrito)
columnas_geo  = ['departamento', 'provincia', 'distrito']
variables_num = data.select_dtypes(include=['number']).columns.tolist()
variables_cat = [
    col for col in data.select_dtypes(include=['object', 'category']).columns
    if col not in columnas_geo
]

# Numéricas: media geográfica, luego media global
data[variables_num] = data.groupby(columnas_geo)[variables_num].transform(
    lambda x: x.fillna(x.mean())
)
data[variables_num] = data[variables_num].fillna(data[variables_num].mean())

# Categóricas: moda geográfica, luego moda global
for col in variables_cat:
    try:
        data[col] = data.groupby(columnas_geo)[col].transform(
            lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x
        )
    except Exception:
        pass

for col in variables_cat:
    if data[col].isna().sum() > 0:
        try:
            data[col] = data[col].fillna(data[col].mode().iloc[0])
        except Exception:
            pass

print('Imputación completada.')

Imputación completada.


In [19]:
# 9.6 Filtro de variabilidad: eliminar columnas con una sola categoría
variables_cat = data.select_dtypes(include=['category', 'object']).columns.tolist()
cols_baja_var = [col for col in variables_cat if data[col].nunique(dropna=True) == 1]
data = data.drop(columns=cols_baja_var)
print(f'Columnas eliminadas por variabilidad=0: {cols_baja_var}')

# 9.7 Filtro de alta cardinalidad (>200 categorías únicas, excepto distrito)
variables_cat = data.select_dtypes(include=['category', 'object']).columns.tolist()
cols_alta_card = [
    col for col in variables_cat
    if data[col].nunique(dropna=True) > 200 and col != 'distrito'
]
data = data.drop(columns=cols_alta_card, errors='ignore')
print(f'Columnas eliminadas por alta cardinalidad: {cols_alta_card}')

Columnas eliminadas por variabilidad=0: ['saldo_obra', 'estado_ejecucion', 'marca_reconstruccion', 'marca_reactivacion', 'obra_reservada', 'nivel_gobierno', 'sector_entidad']
Columnas eliminadas por alta cardinalidad: ['entidad_publica', 'nombre_obra', 'codigo_inversion', 'codigo_snip']


In [20]:
# 9.8 Exclusión por data leakage y variables auxiliares
variables_leakage = [
    'nuevo_plazo_ejecucion',
    'estado_ejecucion',
    'codigo_snip',
    'codigo_inversion',
    'provincia',
    'NOMPRO',
    'brecha_dias',      # leakage directo: brecha_dias > 0 == brecha_existente == 1
    'departamento',     # capturado por Region (macrozona); string de 25 categorías
    'provincia_norm',   # clave auxiliar de merge, no es feature
    'distrito',         # alta cardinalidad, no codificable como string en sklearn
]

# Para Contrata y AD: modalidad_ejecucion es constante, no aporta
if not cfg['incluir_modalidad_como_feature']:
    variables_leakage.append('modalidad_ejecucion')

data = data.drop(columns=variables_leakage, errors='ignore')
print(f'Variables excluidas por leakage/auxiliares: {[v for v in variables_leakage if v in data.columns or True]}')

Variables excluidas por leakage/auxiliares: ['nuevo_plazo_ejecucion', 'estado_ejecucion', 'codigo_snip', 'codigo_inversion', 'provincia', 'NOMPRO', 'brecha_dias', 'departamento', 'provincia_norm', 'distrito']


## 10. Codificación

In [21]:
# 10.1 Dicotómicas: reemplazar texto por 0/1
mapa_dicotomico = {
    'Sí': 1, 'Si': 1, 'sí': 1, 'si': 1, 'Si ': 1,
    'No': 0, 'no': 0, 'NO': 0,
    'Actualizado': 1, 'Desactualizado': 0,
    'Parcial': 1, 'Total': 0,
    'Destacada': 1, 'Emblematicas': 0
}

variables_cat = data.select_dtypes(include=['object', 'category']).columns.tolist()
# brecha_existente excluida del loop: es la variable dependiente, se codifica al final
variables_dic = [col for col in variables_cat if data[col].nunique(dropna=True) == 2 and col != 'brecha_existente']

for col in variables_dic:
    try:
        if hasattr(data[col], 'cat'):  # category dtype: pandas 2.x no admite replace directo
            data[col] = data[col].astype(object)
        data[col] = data[col].replace(mapa_dicotomico)
        data[col] = pd.to_numeric(data[col], errors='coerce')
    except Exception as e:
        print(f'No se pudo convertir {col}: {e}')

# 10.2 Politómicas: one-hot encoding
variables_cat = data.select_dtypes(include=['object', 'category']).columns.tolist()
variables_pol = [
    col for col in variables_cat
    if data[col].nunique(dropna=True) > 2 and col != 'brecha_existente'
]

data = pd.get_dummies(data, columns=variables_pol, drop_first=True)

# Booleanos → 0/1
cols_bool = data.select_dtypes(include=['bool']).columns
data[cols_bool] = data[cols_bool].astype(int)

# 10.3 Encodear variable dependiente explícitamente (después de todo el encoding)
data['brecha_existente'] = (
    data['brecha_existente']
    .astype(str)
    .map({'Sí': 1, 'No': 0, 'sí': 1, 'no': 0, '1': 1, '1.0': 1, '0': 0, '0.0': 0})
    .astype(int)
)

print(f'Shape final: {data.shape}')
print(f'brecha_existente: {data["brecha_existente"].value_counts().to_dict()}')
data.info()

Shape final: (2539, 155)
brecha_existente: {1: 1467, 0: 1072}
<class 'pandas.DataFrame'>
RangeIndex: 2539 entries, 0 to 2538
Columns: 155 entries, n_informes_monitores to tipo_obra_full_Vivienda Construcción Y Saneamiento_Vivienda_Otra Infraestructura
dtypes: float64(6), int32(1), int64(148)
memory usage: 3.0 MB


## 11. [Opcional] Filtro de correlación entre predictores

In [22]:
# Activar con APLICAR_FILTRO_CORRELACION = True
# Elimina variables muy correlacionadas entre sí (>= umbral),
# conservando la que tenga mayor correlación con la variable dependiente.

APLICAR_FILTRO_CORRELACION = True
UMBRAL_CORRELACION         = 0.85

if APLICAR_FILTRO_CORRELACION:
    # brecha_dias y columnas geográficas ya fueron eliminadas en 9.8
    corr_matrix = data.corr(numeric_only=True)
    cor_with_y  = corr_matrix['brecha_existente'].drop('brecha_existente', errors='ignore')
    indep_vars  = cor_with_y.index.tolist()
    corr_indep  = corr_matrix.loc[indep_vars, indep_vars]

    vars_to_remove = set()
    exclusion_log  = []

    for i in range(len(indep_vars)):
        for j in range(i + 1, len(indep_vars)):
            var1, var2 = indep_vars[i], indep_vars[j]
            # omitir pares donde alguna variable ya fue marcada para eliminar
            if var1 in vars_to_remove or var2 in vars_to_remove:
                continue
            r = abs(corr_indep.loc[var1, var2])
            if r >= UMBRAL_CORRELACION:
                cor1, cor2 = abs(cor_with_y[var1]), abs(cor_with_y[var2])
                excluida   = var2 if cor1 >= cor2 else var1
                conservada = var1 if excluida == var2 else var2
                vars_to_remove.add(excluida)
                exclusion_log.append({
                    'var1': var1, 'var2': var2,
                    'correlacion_entre_ellas': r,
                    'cor_var1_con_brecha': cor1,
                    'cor_var2_con_brecha': cor2,
                    'variable_conservada': conservada,
                    'variable_excluida': excluida
                })

    vars_selected = [v for v in indep_vars if v not in vars_to_remove]
    cols_finales  = vars_selected + ['brecha_existente']
    data          = data[cols_finales].copy()

    print(f'Variables eliminadas por correlación: {len(vars_to_remove)}')
    print(f'Shape tras filtro: {data.shape}')

    pd.DataFrame(exclusion_log).to_excel(
        PATH_OUTPUT.replace('.xlsx', '_exclusion_correlacion.xlsx'), index=False
    )
else:
    print('Filtro de correlación no aplicado.')

Variables eliminadas por correlación: 48
Shape tras filtro: (2539, 107)


## 12. Exportación

In [23]:
os.makedirs(os.path.dirname(PATH_OUTPUT), exist_ok=True)
data.to_excel(PATH_OUTPUT, index=False, engine='openpyxl')

print(f'Archivo guardado en: {PATH_OUTPUT}')
print(f'Registros: {len(data):,} | Variables: {data.shape[1]}')
print(f'\nDistribución de brecha_existente:')
print(data['brecha_existente'].value_counts())

Archivo guardado en: C:/15_GFP/data/processed/arcc/1_data_arcc.xlsx
Registros: 2,539 | Variables: 107

Distribución de brecha_existente:
brecha_existente
1    1467
0    1072
Name: count, dtype: int64
